# Medical Document Intelligence System - Primary Notebook

This notebook demonstrates the key functionalities of the Medical Document Intelligence System, including data ingestion, querying, and evaluation.

In [ ]:
import requests
import json
import time

API_BASE = 'http://localhost:5000/api'

## Ingest Sample Medical Data

Ingest sample medical documents into the knowledge base.

In [ ]:
# Sample documents
sample_docs = [
    {
        "name": "Pneumonia Guidelines",
        "content": "PNEUMONIA CLINICAL GUIDELINES\n\nSYMPTOMS\nPneumonia typically presents with fever, cough, chest pain, shortness of breath..."
    },
    {
        "name": "Diabetes Management",
        "content": "TYPE 2 DIABETES MELLITUS MANAGEMENT\n\nSYMPTOMS\nClassic symptoms: Polyuria, Polydipsia, Polyphagia..."
    }
]

# Ingest each document
for doc in sample_docs:
    response = requests.post(f'{API_BASE}/documents/ingest-text', json=doc)
    print(f"Ingested {doc['name']}: {response.status_code}")

print("Sample data ingested.")

## Run a Query

Query the system with a medical question.

In [ ]:
query = "What are the symptoms of pneumonia?"

response = requests.post(f'{API_BASE}/query', json={'query': query})
result = response.json()

print("Query:", query)
print("RAG Answer:", result.get('answer', 'N/A'))
print("Sources:", result.get('sources', []))

## Compare RAG vs Baseline

Compare the RAG response with a baseline LLM response.

In [ ]:
comparison = requests.post(f'{API_BASE}/compare', json={'query': query})
comp_result = comparison.json()

print("RAG Answer:", comp_result['rag']['answer'])
print("Baseline Answer:", comp_result['baseline']['answer'])
print("RAG Time:", comp_result['rag']['processingTimeMs'], "ms")
print("Baseline Time:", comp_result['baseline']['processingTimeMs'], "ms")

## Evaluation

Run evaluation on test queries.

In [ ]:
# Test queries
test_queries = [
    "What are the symptoms of pneumonia?",
    "How do you diagnose type 2 diabetes?",
    "What is the first-line treatment for hypertension?"
]

results = []
for q in test_queries:
    comp = requests.post(f'{API_BASE}/compare', json={'query': q}).json()
    results.append({
        'query': q,
        'rag_time': comp['rag']['processingTimeMs'],
        'baseline_time': comp['baseline']['processingTimeMs']
    })
    time.sleep(1)  # Avoid overwhelming the server

# Display results
import pandas as pd
df = pd.DataFrame(results)
print(df)